In [1]:
# ============================================================
# STEP 4: WHISPER BASELINE + WER (Colab, ek cell)
# ============================================================
import os, sys, time

# ---------- 1. Repo ka latest code ----------
REPO_URL = "https://github.com/FatimaaShaheen/urdu-asr-subtitles.git"
REPO_DIR = "/content/urdu-asr-subtitles"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# ---------- 2. Libraries ----------
!pip install -q "datasets<4" transformers accelerate jiwer librosa soundfile

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

# ---------- 3. Imports ----------
import torch, pandas as pd
from transformers import pipeline
from src.data.load_dataset import load_fleurs_urdu
from src.evaluation.metrics import compute_metrics, normalize_urdu

device = 0 if torch.cuda.is_available() else -1
print("GPU:", torch.cuda.get_device_name(0) if device == 0 else "NAHI (CPU, bohat slow)")

# ---------- 4. Poora test set (299 examples) ----------
test = load_fleurs_urdu("test")
audios = [s["audio"]["array"] for s in test]
refs = list(test["transcription"])
print("Test examples:", len(test))

# ---------- 5. Model chalao aur WER naapo ----------
def run_baseline(model_name):
    asr = pipeline(
        "automatic-speech-recognition",
        model=model_name,
        device=device,
        torch_dtype=torch.float16 if device == 0 else torch.float32,
        chunk_length_s=30,   # test mein 30s se lambi audio bhi hai
    )
    t0 = time.time()
    outs = asr(
        audios,
        batch_size=8,
        generate_kwargs={"language": "urdu", "task": "transcribe"},
    )
    secs = time.time() - t0
    hyps = [o["text"] for o in outs]
    m = compute_metrics(refs, hyps)
    m.update({"model": model_name, "seconds": round(secs, 1)})
    del asr
    torch.cuda.empty_cache()
    return m, hyps

results, all_hyps = [], {}
for name in ["openai/whisper-base", "openai/whisper-small"]:
    print("\nChal raha hai:", name)
    m, hyps = run_baseline(name)
    results.append(m)
    all_hyps[name] = hyps
    print(m)

# ---------- 6. Natija ka table ----------
print("\n=== BASELINE RESULTS (fine-tuning se pehle) ===")
print(pd.DataFrame(results)[["model", "wer", "cer", "n", "seconds"]])

# ---------- 7. Reference vs prediction ----------
hyps = all_hyps["openai/whisper-small"]
print("\n=== whisper-small: pehle 8 examples ===")
for r, h in list(zip(refs, hyps))[:8]:
    print("REF :", normalize_urdu(r))
    print("PRED:", normalize_urdu(h))
    print("-" * 60)

# ---------- 8. CSV files download ----------
pd.DataFrame({"reference": refs, "prediction": hyps}).to_csv(
    "baseline_small_predictions.csv", index=False)
pd.DataFrame(results).to_csv("baseline_results.csv", index=False)
from google.colab import files
files.download("baseline_small_predictions.csv")
files.download("baseline_results.csv")

Cloning into '/content/urdu-asr-subtitles'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 52 (delta 16), reused 43 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 2.36 MiB | 13.13 MiB/s, done.
Resolving deltas: 100% (16/16), done.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
GPU: Tesla T4


README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

parquet-data/ur_pk/train-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 1.60GB            

parquet-data/ur_pk/train-00000-of-00001.(…): downloading bytes:           |  0.00B            

parquet-data/ur_pk/validation-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  174MB            

parquet-data/ur_pk/validation-00000-of-0(…): downloading bytes:           |  0.00B            

... (37 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>